Generative AI to improve upon customer insights about their spending and budgeting.
1. Overdraft Predictive Notification: Warn users based on their history of spending and their current spending trends if they are at risk of overdraft with future scheduled or historical payments (subscriptions, bills, etc)
2. Integration with a Gen AI chat bot for specific insights - Generative Summary report based on user input *


In [ ]:
!pip install -q transformers==4.4.2

!pip install torch-scatter -f https://pytorch-geometric.com/whl/torch-1.8.0+cu101.html


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  error: subprocess-exited-with-error
  
  × Building wheel for tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for tokenizers
ERROR: Could not build wheels for tokenizers, which is required to install pyproject.toml-based projects
Looking in links: https://pytorch-geometric.com/whl/torch-1.8.0+cu101.html


In [17]:
import torch
from google.colab import drive
from transformers import pipeline, TapasTokenizer, TapasForQuestionAnswering
import pandas as pd
import random
from dateutil import parser
from ipywidgets import widgets, Layout, VBox, HTML
from IPython.display import display, HTML
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [4]:
drive.mount('/content/gdrive', force_remount=True)


Mounted at /content/gdrive


In [6]:
table = pd.read_csv("/content/gdrive/MyDrive/bankStatmentJanSmall.csv")
table = table.astype(str)

In [ ]:
table

,Name,Amount,Category,Transaction Type,Date,Account Balance
0,Bi-weekly Paycheck,2500,Paychecks Salary,Income,1/1/2024,2500
1,Rent,1600,Home and Utilities,Spending,1/5/2024,900
2,Kroger,150,Groceries,Spending,1/7/2024,750
3,Clothing,50,Shopping and Entertainment,Spending,1/7/2024,700
4,Work Expense Reimbursement,75,Expense Reimbursement,Income,1/9/2024,775
5,Takeout,20,Restaurants and Dining,Spending,1/12/2024,755
6,Car Payment,350,Transportation,Spending,1/12/2024,405
7,Student Debt Payment,150,Education,Spending,1/12/2024,255
8,Bi-weekly Paycheck,2500,Paychecks Salary,Income,1/15/2024,2755
9,Shake Shack,15,Restaurants and Dining,Spending,1/17/2024,2740


In [7]:
model_name = "google/tapas-base-finetuned-wtq"
# load the tokenizer and the model from huggingface model hub
tokenizer = TapasTokenizer.from_pretrained(model_name)
model = TapasForQuestionAnswering.from_pretrained(model_name, local_files_only=False)
# load the model and tokenizer into a question-answering pipeline
pipe = pipeline("table-question-answering",  model=model, tokenizer=tokenizer, device=device)


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:88: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/490 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/262k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/154 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.66k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/443M [00:00<?, ?B/s]

In [9]:
#Helper Functions
def process_qa_output(qa_output, question):
    if "day" in question:
        # If yes, call get_day_of_week function and return the day of the week
        qa_output['answer'] = get_day_of_week(qa_output['answer'])

    if 'aggregator' in qa_output and qa_output['aggregator'] == 'COUNT':
        qa_output['answer'] = count_helper(qa_output["answer"])
    return qa_output


def get_day_of_week(date_str):
    try:
        date_obj = parser.parse(date_str)
        day_of_week = date_obj.weekday()
        days = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
        return days[day_of_week]
    except ValueError:
        return "Invalid Date Format"

def count_helper(input_string):
    parts = input_string.split('>')

    count = 0
    for part in parts[1:]:
        count += len(part.split(','))

    return count


In [8]:
questionBank = ["What day of the week did I have the most spending this month?",
                "How many times did I buy Starbucks in January?",
]

In [10]:
print(questionBank[0])
query=questionBank[0]
qa_output = pipe(table=table, query=query)
print(process_qa_output(qa_output, query))

What day of the week did I have the most spending this month?
{'answer': 'Friday', 'coordinates': [(1, 4)], 'cells': ['1/5/2024'], 'aggregator': 'NONE'}


In [11]:
print(questionBank[1])
query=questionBank[1]
qa_output = pipe(table=table, query=query)
print(process_qa_output(qa_output, query))

How many times did I buy Starbucks in January?
{'answer': 2, 'coordinates': [(13, 5), (18, 5)], 'cells': ['2425', '2095'], 'aggregator': 'COUNT'}


In [20]:
from IPython.display import display, Markdown

class ConversationApp:
    def __init__(self):
        self.conversation_output = []
        self.send_message("Hello!")

    def send_message(self, message):
        if message:
            user_message = f"You: {message}"
            bot_message = "Bot: I'm just a bot, I can't respond!"
            self.conversation_output.append(user_message)
            self.conversation_output.append(bot_message)
            display(Markdown('\n\n'.join(self.conversation_output)))

app = ConversationApp()

You: Hello!

Bot: I'm just a bot, I can't respond!